# Integridad

Verifica versiones, hashes, intentos y participantes. Los datos ausentes permanecen desconocidos; ningún contador se estima desde caracteres.

In [ ]:
from pathlib import Path
import json, os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from native_eval.bundle import COLUMNS, comparison, verify_bundle
bundle = Path(globals().get('BUNDLE', os.environ.get('NATIVE_EVAL_BUNDLE', '.runs/bundle')))
runs = pd.read_csv(bundle / 'runs.csv') if (bundle / 'runs.csv').is_file() else pd.DataFrame(columns=COLUMNS)
print('Sin resultados: no hay corridas de evaluación.' if runs.empty else f'{len(runs)} intentos observados; se muestran también los fallos.')


In [ ]:
if (bundle / 'manifest.json').exists():
    manifest = verify_bundle(bundle)
    campaign = json.loads((bundle / 'campaign.json').read_text())
    display({k: campaign[k] for k in ('protocol','benchmark_commit','harbor','claude_code','model','effort','seed')})
    print(f"{len(manifest['files'])} archivos verificados")
    expected = {s['slot_id'] for s in campaign['schedule']}
    observed = set(runs['slot_id'])
    display({'planned':len(expected), 'observed':len(observed), 'missing':sorted(expected-observed), 'unexpected':sorted(observed-expected)})
if not runs.empty:
    display(runs[['slot_id','accounting_complete','protocol_ok','reward','success','incident_count']])
    calls = pd.DataFrame(json.loads((bundle / 'calls.json').read_text()))
    display(calls)
    display(pd.DataFrame(json.loads((bundle / 'incidents.json').read_text())))
